# Calibration-free MI-BCI on FPGA — experiment notebook (v2)

**Runtime → Change runtime type → T4 GPU.**  Put `hanet_mi.zip` in `MyDrive/FPGABCI/` (delete older copies first).

Everything (dataset, cache, results) is saved on Drive. If Colab disconnects: run cells 1–3 again, then the cell you were on — finished work is skipped automatically.

In [ ]:
# 1. Drive + code
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/FPGABCI'
for d in ('runs', 'mne_data', 'cache', 'pretrain'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
!unzip -oq {BASE}/hanet_mi.zip -d /content/
%cd /content/hanet_mi/python
!python train_loso.py -h | grep -E -c 'init|cache-dir|P1|act'   # must print 4 (new code loaded)

In [ ]:
# 2. Install
!pip install -q moabb
import mne
mne.set_config('MNE_DATA', f'{BASE}/mne_data', set_env=True)
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
# 3. Helpers (run once per session)
import subprocess, sys
PROP = '--no-attention --K1 64 --P1 4 --P2 8 --act elu --dg none'   # proposed HW-friendly CNN
ATTN = '--K1 64 --P1 4 --P2 8 --act elu --dg none'                  # + hardware attention (ablation)
EEG  = '--model eegnet --dg none'
SHAL = '--model shallow --dg none'
CACHE = f'--cache-dir {BASE}/cache'

def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for l in p.stdout:   # live progress
        if l.startswith(('S0', '[init]', '[data]', '[pretrain]', 'saved', 'loaded', 'train ', 'ep ',
                         'target', 'fold', 'exported', 'Traceback')) or 'error' in l.lower():
            print(l, end='', flush=True)
    p.wait()

def run(name, extra, dataset='2a', subjects=range(1, 10)):
    """LOSO over all subjects; skips subjects already finished."""
    for s in subjects:
        out = f'{BASE}/runs/{name}/S{s}'
        if os.path.exists(f'{out}/results.json'):
            continue
        sh(f'python -W ignore train_loso.py --dataset {dataset} --targets {s} --out {out} {CACHE} {extra}')
    print(f'== {name} done')

def pretrain(name, extra, classes='all'):
    out = f'{BASE}/pretrain/{name}.pt'
    if os.path.exists(out):
        print(f'{out} exists, skipping'); return out
    sh(f'python -W ignore pretrain.py --out {out} --classes {classes} {CACHE} {extra}')
    return out

## 4. Priority 1 — the key question: does PhysioNet pretraining help? (~1.5–2 h)
The first call downloads and preprocesses PhysioNet (~2 GB, 20–40 min, once; cached on Drive afterwards).

In [ ]:
# N = how many PhysioNet subjects to use (fast: those already downloaded). 0 = all 104.
N = 50
pt_prop = pretrain(f'physio_prop_n{N}', f'{PROP} --n-subjects {N} --epochs 40 --patience 10')
run(f'B4_prop_pt{N}_s42', f'{PROP} --init {pt_prop} --seed 42 --epochs 80 --patience 20')

## 5. Summary of any runs (edit the groups)

In [ ]:
!python aggregate.py --base {BASE}/runs --metric online_ea \
   --group "Prop+PT=B4_prop_pt_s42" --group "Prop=A4_prop_s42" --group "EEGNet=A4_eegnet_s42"

## 6. Priority 2 — seeds for the main table (4-class) (~2–3 h)

In [ ]:
for sd in (7, 123):
    run(f'B4_prop_pt_s{sd}', f'{PROP} --init {pt_prop} --seed {sd}')
    run(f'A4_eegnet_s{sd}',  f'{EEG} --seed {sd}')
    run(f'A4_prop_s{sd}',    f'{PROP} --seed {sd}')

## 7. Priority 3 — 2-class (left vs right hand) (~1.5 h)

In [ ]:
pt_prop_lr = pretrain('physio_prop_lr', PROP, classes='lr')
for sd in (42, 7, 123):
    run(f'B2_prop_pt_s{sd}', f'{PROP} --classes lr --init {pt_prop_lr} --seed {sd}')
    run(f'A2_eegnet_s{sd}',  f'{EEG} --classes lr --seed {sd}')

## 8. Priority 4 — ablations & extra baseline (~2 h)

In [ ]:
pt_attn = pretrain('physio_attn', ATTN)
run('B4_attn_pt_s42', f'{ATTN} --init {pt_attn} --seed 42')     # + attention
run('B4_prop_pt_noreset_s42', f'{PROP} --init {pt_prop} --seed 42 --no-reset-session')  # online-EA without session reset
run('A4_shallow_s42', f'{SHAL} --seed 42')

## 9. Final tables (paste the output to Claude)

In [ ]:
G4 = ['--group "Prop+PT=B4_prop_pt_s42,B4_prop_pt_s7,B4_prop_pt_s123"',
      '--group "Prop=A4_prop_s42,A4_prop_s7,A4_prop_s123"',
      '--group "EEGNet=A4_eegnet_s42,A4_eegnet_s7,A4_eegnet_s123"',
      '--group "ShallowConvNet=A4_shallow_s42"',
      '--group "Prop+Attn+PT=B4_attn_pt_s42"']
for m in ('online_ea', 'offline_ea_oracle', 'no_ea'):
    !python aggregate.py --base {BASE}/runs --metric {m} {' '.join(G4)} --csv {BASE}/table4_{m}.csv
G2 = ['--group "Prop+PT=B2_prop_pt_s42,B2_prop_pt_s7,B2_prop_pt_s123"', '--group "EEGNet=A2_eegnet_s42,A2_eegnet_s7,A2_eegnet_s123"']
!python aggregate.py --base {BASE}/runs --metric online_ea {' '.join(G2)} --csv {BASE}/table2_online.csv

## 10. int8 quantisation + HLS export of the final model (per subject)

In [ ]:
for s in range(1, 10):
    sh(f'python -W ignore deploy.py --ckpt {BASE}/runs/B4_prop_pt_s42/S{s}/hanet_target{s}.pt --dataset 2a --target {s} --hls-dir {BASE}/hls_export/S{s}')
# prints: float vs int8 accuracy per subject -> quantisation table